## Pseudo-bolometric luminosity vs time

For each FINAL spectrum epoch we compute a **pseudo-bolometric flux** by integrating linear spectral flux density over **fixed** observer-frame wavelength bounds \([\lambda_{\min}, \lambda_{\max}]\) (Å):

$$F_{\mathrm{pseudo}}(t) = \int_{\lambda_{\min}}^{\lambda_{\max}} F_\lambda(\lambda,t)\,d\lambda$$

with \(F_\lambda\) in erg s\(^{-1}\) cm\(^{-2}\) Å\(^{-1}\) after `read_final_spectrum_linear`, so \(F_{\mathrm{pseudo}}\) is in **erg s\(^{-1}\) cm\(^{-2}\)**.

Then

$$L_{\mathrm{pseudo}}(t) = 4\pi D_L^2\, F_{\mathrm{pseudo}}(t)$$

with \(D_L\) in **cm** ⇒ \(L\) in **erg s\(^{-1}\)**.

**Distance:** By default \(D_L\) comes from **`astropy.cosmology.Planck18.luminosity_distance(z)`** with \(z\) read from [`Inputs/SNe_Info/info.dat`](../Inputs/SNe_Info/info.dat) for your `SNNAME`. Set `LUMINOSITY_DISTANCE_MPC` to use a manual distance (e.g. a GW–EM joint value) instead; cosmological \(D_L(z)\) and siren distances for GW170817 can differ at roughly the 10% level.

**Coverage:** The same \(\lambda_{\min}, \lambda_{\max}\) must apply to every epoch. If an epoch’s spectrum does not cover the full interval (strict mode), that point is **NaN** so the light curve does not gain fake jumps from shrinking the integral domain. **Intersection** mode instead sets the bounds to the common wavelength overlap across all loaded spectra (still fixed in time).

**Flux uncertainties:** We report an approximate \(1\sigma\) error on \(F_{\mathrm{pseudo}}\) as \(\sqrt{\int \sigma_\lambda^2 \, d\lambda}\) over the same \(\lambda\) window (trapezoid on \(\sigma_\lambda^2\)), then scale to \(L\) with \(4\pi D_L^2\).

**Paths / MJD mapping:** Matches [`7.5_alternate.ipynb`](7.5_alternate.ipynb) (`resolve_final_directory`, `read_final_spectrum_linear`, `stem_to_spec_mjd`).

**Synthetic curve (plot cell):** `SYNTH_LINE_MODE` selects `"none"`, `"gp"` (`smooth_lightcurve_gp` on linear \(L\), then log display if `LOG_L_AXIS`), `"smooth_trapz"` (phase vs plotted \(y\) = \(L\) or \(\log_{10} L\) with `UnivariateSpline`, same as 7.5), or `"trapz_linear"` (piecewise-linear / “spliced” segments through the epochs). Set **`SHOW_SYNTHETIC_POINTS = False`** to plot only the curve with no scatter + error bars.

In [1]:
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime

rt = pconf.bootstrap_runtime(photometry_stage="extrapolated")
OUTPUT_DIR = rt.output_dir
OUTPUT_PATH = rt.output_path
DATASPEC_PATH = rt.dataspec_path
DATAINFO_PATH = rt.datainfo_path
FILTER_PATH = rt.filter_path
FILTER_LEAF = rt.filter_leaf
FILTERS_PARENT = rt.filters_parent
color_dict, mark_dict, exclude_filt = rt.color_dict, rt.mark_dict, rt.exclude_filt
FINAL_SPECTRA_DIR = rt.final_spectra_dir
GP_MODE = rt.gp_mode
import what_the_flux as wtf

import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import UnivariateSpline
from astropy import units as u
from astropy.cosmology import Planck18
    import george  # noqa: F401
from comparison_check_log_utils import (
from comparison_trapz_lc import smooth_lightcurve_gp


In [ ]:
USE_ITER_GP_MANGLE_FINAL = True  # from pipeline_config
# --- Mirror 7.5_alternate-style configuration ---
COCO_PATH = pconf.COCO_PATH
SNNAME = "AT2017gfo"
DATALC_PATH = os.path.join(COCO_PATH, "Inputs", "Photometry", "3_LCs_extrapolated")
FINAL_FLUX_ON_DISK = "auto"
FINAL_VARIANT = "as_observed"
FINAL_SUFFIXES_TO_LOAD = None  # or e.g. ("_FINAL_spec_FL.txt",)
FINAL_DATA_DIR = pconf.final_spectra_qa_dir(pconf.outputs_root(COCO_PATH), SNNAME)

# Wavelength window (observer / file frame, Å)
WL_MIN_A = 3257.341  
WL_MAX_A = 24276.87
# "strict": NaN if spectrum does not cover [WL_MIN_A, WL_MAX_A]; "intersection": set bounds to common overlap of all spectra
WL_COVERAGE_MODE = "strict"
EDGE_TOL_A = 0.5  # Å tolerance when checking spectrum edges vs integration limits

# Luminosity distance: None => use Planck18 from catalog z; else Mpc (overrides cosmology for D_L only)
LUMINOSITY_DISTANCE_MPC = None
COSMOLOGY = Planck18

# Phase zero (MJD)
PHASE_REFERENCE_MJD = None  # None -> pconf.SN_EXPLOSION_MJD[SNNAME]

# Plotting (7.5_alternate-style)
FIGSIZE = (9, 6)
LOG_L_AXIS = False
LOG_TIME_AXIS = False
LOG_TIME_LINTHRESH_DAYS = 0.1
FIG_SAVE_PATH = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/pseudo-bolometric-luminosity-onlylc_ryanversion.png'  # e.g. os.path.join(pconf.outputs_root(COCO_PATH), SNNAME, "pseudo_Lbol.png")
FIG_SAVE_DPI = 500
LINE_WIDTH = 2.2

# Synthetic curve / points (same modes as 7.5_alternate: gp | smooth_trapz | trapz_linear | none)
SYNTH_LINE_MODE = "smooth_trapz"
TRAPZ_SPLINE_S = 0
N_TRAPZ_LINE = 128
SHOW_SYNTHETIC_POINTS = False

# GP knobs (only used when SYNTH_LINE_MODE == "gp")
GP_LENGTH_SCALE_DAYS = None
GP_LENGTH_SCALE_FACTOR = 1.0
GP_OPTIMIZE = True
N_GP_GRID = 400

data_dir = resolve_final_directory(
    COCO_PATH, SNNAME, FINAL_VARIANT, twodim_branch=TWODIM_BRANCH
)
print("FINAL dir:", data_dir)


In [ ]:
def read_sn_redshift(coco_path: str, snname: str) -> float:
    path = os.path.join(coco_path, "Inputs", "SNe_Info", "info.dat")
    if not os.path.isfile(path):
        raise FileNotFoundError("Missing %s" % path)
    df = pd.read_csv(path, sep=r"\s+", comment="#")
    if "Name" not in df.columns or "z" not in df.columns:
        raise ValueError("info.dat must have Name and z columns")
    row = df.loc[df["Name"] == snname]
    if len(row) != 1:
        raise ValueError("Expected exactly one row for Name=%r in %s" % (snname, path))
    return float(row.iloc[0]["z"])


def luminosity_distance_cm(*, z: float | None, d_l_mpc: float | None, cosmology) -> tuple[float, str]:
    if d_l_mpc is not None:
        d_cm = (d_l_mpc * u.Mpc).to(u.cm).value
        return d_cm, "manual LUMINOSITY_DISTANCE_MPC = %s" % d_l_mpc
    if z is None:
        raise ValueError("Need z or LUMINOSITY_DISTANCE_MPC")
    d_cm = cosmology.luminosity_distance(z).to(u.cm).value
    cname = getattr(cosmology, "name", type(cosmology).__name__)
    return d_cm, "%s.luminosity_distance(z=%.6g)" % (cname, z)


def trapz_pseudo_flux(
    wl: np.ndarray,
    fl: np.ndarray,
    fe: np.ndarray,
    lam_min: float,
    lam_max: float,
    *,
    strict_cover: bool,
    edge_tol: float,
) -> tuple[float, float]:
    """Return (F_pseudo, sigma_F) in erg/s/cm^2; NaNs if invalid / strict fail."""
    m = np.isfinite(wl) & np.isfinite(fl) & np.isfinite(fe)
    wl, fl, fe = wl[m], fl[m], fe[m]
    if wl.size < 2:
        return float("nan"), float("nan")
    o = np.argsort(wl)
    wl, fl, fe = wl[o], fl[o], fe[o]
    if strict_cover:
        wmin, wmax = float(np.min(wl)), float(np.max(wl))
        if wmin > lam_min + edge_tol or wmax < lam_max - edge_tol:
            return float("nan"), float("nan")
    idx = (wl >= lam_min) & (wl <= lam_max)
    if np.count_nonzero(idx) < 2:
        return float("nan"), float("nan")
    w = wl[idx]
    f = fl[idx]
    e = fe[idx]
    F = float(np.trapz(f, w))
    var = float(np.trapz(e**2, w))
    sig = float(np.sqrt(max(var, 0.0)))
    return F, sig


z_cat = read_sn_redshift(COCO_PATH, SNNAME)
D_L_cm, dl_note = luminosity_distance_cm(
    z=z_cat, d_l_mpc=LUMINOSITY_DISTANCE_MPC, cosmology=COSMOLOGY
)
print("Catalog z:", z_cat)
print("D_L:", dl_note)
print("D_L (cm):", D_L_cm)
print("4π D_L² (cm²):", 4.0 * np.pi * D_L_cm**2)

t0 = float(
    PHASE_REFERENCE_MJD
    if PHASE_REFERENCE_MJD is not None
    else pconf.SN_EXPLOSION_MJD[SNNAME]
)
print("Phase reference MJD (t0):", t0)

spectra_files = sorted(f for f in os.listdir(data_dir) if f.endswith(".txt"))
if FINAL_SUFFIXES_TO_LOAD:
    spectra_files = [f for f in spectra_files if any(f.endswith(s) for s in FINAL_SUFFIXES_TO_LOAD)]
if not spectra_files:
    raise FileNotFoundError("No matching .txt in %s" % data_dir)

coverage_rows = []
prepared = []
for fname in spectra_files:
    path = os.path.join(data_dir, fname)
    wl, fl, fe = read_final_spectrum_linear(path, flux_on_disk=FINAL_FLUX_ON_DISK)
    m = np.isfinite(wl) & np.isfinite(fl) & np.isfinite(fe)
    wl, fl, fe = wl[m], fl[m], fe[m]
    if wl.size < 2:
        continue
    order = np.argsort(wl)
    wl, fl, fe = wl[order], fl[order], fe[order]
    wl, fl, fe = deduplicate_wavelength_flux(wl, fl, fe)
    stem_f = parse_final_stem(fname)
    smjd = stem_to_spec_mjd(stem_f, COCO_PATH, SNNAME, datalc_path=DATALC_PATH)
    prepared.append(
        (float(smjd), fname, np.asarray(wl), np.asarray(fl), np.asarray(fe))
    )
    coverage_rows.append(
        {"fname": fname, "min_A": float(wl.min()), "max_A": float(wl.max()), "n_pix": int(wl.size)}
    )

prepared.sort(key=lambda r: (r[0], r[1]))
cov_df = pd.DataFrame(coverage_rows)
if cov_df.empty:
    raise ValueError("No valid spectra")
print("Wavelength coverage (per file):")
with pd.option_context("display.max_rows", 12):
    print(cov_df)

if WL_COVERAGE_MODE not in ("strict", "intersection"):
    raise ValueError("WL_COVERAGE_MODE must be 'strict' or 'intersection'")

if WL_COVERAGE_MODE == "intersection":
    lam_min_eff = float(cov_df["min_A"].max())
    lam_max_eff = float(cov_df["max_A"].min())
    if not (lam_min_eff < lam_max_eff):
        raise ValueError("Intersection λ range is empty; check spectra overlap")
    print(
        "Intersection bounds (constant across epochs): %.3f – %.3f Å"
        % (lam_min_eff, lam_max_eff)
    )
else:
    lam_min_eff, lam_max_eff = float(WL_MIN_A), float(WL_MAX_A)
    print(
        "User bounds (strict): %.3f – %.3f Å" % (lam_min_eff, lam_max_eff)
    )

strict_flag = WL_COVERAGE_MODE == "strict"

four_pi_r2 = 4.0 * np.pi * D_L_cm**2
records = []
for smjd, fname, wl, fl, fe in prepared:
    F, sigF = trapz_pseudo_flux(
        wl,
        fl,
        fe,
        lam_min_eff,
        lam_max_eff,
        strict_cover=strict_flag,
        edge_tol=EDGE_TOL_A,
    )
    if np.isfinite(F):
        L = four_pi_r2 * F
        Lerr = four_pi_r2 * sigF if np.isfinite(sigF) else float("nan")
    else:
        L = Lerr = float("nan")
    records.append(
        {
            "mjd": smjd,
            "phase_days": smjd - t0,
            "fname": fname,
            "F_pseudo_erg_s_cm2": F,
            "F_pseudo_err": sigF,
            "L_erg_s": L,
            "L_err": Lerr,
        }
    )

bolo = pd.DataFrame(records).sort_values("mjd").reset_index(drop=True)
n_nan = int(np.isnan(bolo["F_pseudo_erg_s_cm2"].values).sum())
if n_nan:
    warnings.warn(
        "%i / %i epochs have NaN pseudo-flux (insufficient λ coverage for chosen bounds)"
        % (n_nan, len(bolo))
    )
    print("Epochs with NaN pseudo-flux:")
    print(bolo.loc[np.isnan(bolo["F_pseudo_erg_s_cm2"]), ["mjd", "phase_days", "fname"]])
print("Bolometric table (head):")
print(bolo.head(10))


In [7]:
SHOW_SYNTHETIC_POINTS = False

In [ ]:
def _maybe_save_fig(fig, path):
    if path is None or str(path).strip() == "":
        return
    from pathlib import Path

    p = Path(path).expanduser()
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, dpi=int(FIG_SAVE_DPI), bbox_inches="tight")
    print("Saved figure ->", p)


_VALID_SYNTH_MODES = frozenset({"none", "gp", "smooth_trapz", "trapz_linear"})
if SYNTH_LINE_MODE not in _VALID_SYNTH_MODES:
    raise ValueError(
        "SYNTH_LINE_MODE must be one of %s, got %r"
        % (sorted(_VALID_SYNTH_MODES), SYNTH_LINE_MODE)
    )

if SYNTH_LINE_MODE == "none" and not SHOW_SYNTHETIC_POINTS:
    warnings.warn(
        "Nothing to plot: SYNTH_LINE_MODE is 'none' and SHOW_SYNTHETIC_POINTS is False"
    )


def _trapz_sorted_finite(syn_phase, yy, s_sort):
    ph = syn_phase[s_sort]
    val = yy[s_sort]
    m = np.isfinite(ph) & np.isfinite(val)
    ph, val = ph[m], val[m]
    if ph.size == 0:
        return None, None
    o = np.argsort(ph)
    return ph[o], val[o]


def _uniq_phase_avg_y(ph, val):
    if ph.size <= 1:
        return ph, val
    u_ph, inv = np.unique(ph, return_inverse=True)
    if u_ph.size == ph.size:
        return ph, val
    wsum = np.bincount(inv, weights=val)
    cnt = np.bincount(inv)
    return u_ph, wsum / cnt


def _smooth_trapz_line(syn_phase, yy, s_sort):
    ph, val = _trapz_sorted_finite(syn_phase, yy, s_sort)
    if ph is None or ph.size < 2:
        return None, None
    ph, val = _uniq_phase_avg_y(ph, val)
    if ph.size < 2:
        return None, None
    n = int(max(2, N_TRAPZ_LINE))
    tgrid = np.linspace(float(ph[0]), float(ph[-1]), n)
    if ph.size == 2:
        val_g = np.interp(tgrid, ph, val)
        return tgrid, val_g
    k = int(min(3, ph.size - 1))
    spl = UnivariateSpline(ph, val, s=TRAPZ_SPLINE_S, k=k)
    return tgrid, spl(tgrid)


def _trapz_linear_line(syn_phase, yy, s_sort):
    ph, val = _trapz_sorted_finite(syn_phase, yy, s_sort)
    if ph is None or ph.size < 2:
        return None, None
    ph, val = _uniq_phase_avg_y(ph, val)
    if ph.size < 2:
        return None, None
    return ph, val


plt.rcParams["font.family"] = "serif"
fig, ax = plt.subplots(figsize=FIGSIZE)

ph = bolo["phase_days"].values.astype(float)
L = bolo["L_erg_s"].values.astype(float)
Le = bolo["L_err"].values.astype(float)
mjd_sorted = bolo["mjd"].values.astype(float)
s = np.argsort(ph)
ph, L, Le, mjd_sorted = ph[s], L[s], Le[s], mjd_sorted[s]

mfin = np.isfinite(ph) & np.isfinite(L) & (L > 0)
err_lo = err_hi = None
if LOG_L_AXIS:
    y = np.full_like(L, np.nan, dtype=float)
    y[mfin] = np.log10(L[mfin])
    ylabel = r"$\log_{10}(L_{\mathrm{pseudo}}\, /\, \mathrm{erg\,s^{-1}})$"
    merr = mfin & np.isfinite(Le) & (Le > 0) & (L > Le)
    if np.any(merr):
        elow = np.full_like(L, np.nan, dtype=float)
        ehigh = np.full_like(L, np.nan, dtype=float)
        elow[merr] = y[merr] - np.log10(L[merr] - Le[merr])
        ehigh[merr] = np.log10(L[merr] + Le[merr]) - y[merr]
        err_lo, err_hi = elow, ehigh
else:
    y = L.astype(float)
    ylabel = r"$L$ (erg s$^{-1}$)"

msk = np.isfinite(ph) & np.isfinite(y)
s_sort = np.argsort(ph)
line_drawn = False

# Underlying synthetic curve (zorder below points)
if SYNTH_LINE_MODE == "gp":
    if george is None:
        warnings.warn(
            "george not installed; set SYNTH_LINE_MODE to smooth_trapz/trapz_linear/none or pip install george"
        )
    else:
        mgp = mfin & np.isfinite(Le) & (Le > 0) & np.isfinite(mjd_sorted)
        t_mjd = mjd_sorted[mgp]
        yL = L[mgp]
        yeL = np.maximum(Le[mgp], 1e-6 * yL)
        if t_mjd.size >= 3:
            t_lo, t_hi = float(np.min(t_mjd)), float(np.max(t_mjd))
            t_grid = np.linspace(t_lo, t_hi, int(N_GP_GRID))
            mu, _std = smooth_lightcurve_gp(
                t_mjd,
                yL,
                yeL,
                t_grid,
                optimize=GP_OPTIMIZE,
                length_scale_days=GP_LENGTH_SCALE_DAYS,
                length_scale_factor=GP_LENGTH_SCALE_FACTOR,
            )
            phase_grid = t_grid - t0
            mu_plot = np.log10(np.maximum(mu, 1e-300)) if LOG_L_AXIS else mu
            ax.plot(
                phase_grid,
                mu_plot,
                "-",
                color="crimson",
                lw=LINE_WIDTH,
                alpha=0.75,
                zorder=2,
                label="GP smooth ($L$)",
            )
            line_drawn = True
        else:
            print("GP skipped: need >=3 finite points with L>0 and L_err>0")
elif SYNTH_LINE_MODE == "smooth_trapz":
    lx, ly = _smooth_trapz_line(ph, y, s_sort)
    if lx is not None:
        ax.plot(
            lx,
            ly,
            "-",
            color="deepskyblue",
            lw=LINE_WIDTH,
            alpha=0.75,
            zorder=2,
            label="Pseudo-bolometric luminosity",
        )
        line_drawn = True
    else:
        print("Spline smooth skipped: need >=2 finite phase–luminosity points")
elif SYNTH_LINE_MODE == "trapz_linear":
    lx, ly = _trapz_linear_line(ph, y, s_sort)
    if lx is not None:
        ax.plot(
            lx,
            ly,
            "-",
            color="crimson",
            lw=LINE_WIDTH,
            alpha=0.75,
            zorder=2,
            label="Piecewise linear",
        )
        line_drawn = True
    else:
        print("Piecewise linear skipped: need >=2 finite phase–luminosity points")

if SHOW_SYNTHETIC_POINTS:
    if LOG_L_AXIS and err_lo is not None and err_hi is not None:
        _yerr = (err_lo[msk], err_hi[msk])
    elif not LOG_L_AXIS:
        _yerr = Le[msk]
    else:
        _yerr = None
    if np.any(msk):
        ax.errorbar(
            ph[msk],
            y[msk],
            yerr=_yerr,
            fmt="o",
            linestyle="none",
            markersize=7,
            markerfacecolor="white",
            markeredgecolor="darkred",
            markeredgewidth=1.2,
            ecolor="darkred",
            elinewidth=1.2,
            capsize=2.5,
            capthick=1.2,
            alpha=1.0,
            zorder=10,
            label="synthetic epochs",
        )
elif not line_drawn and SYNTH_LINE_MODE != "none":
    warnings.warn(
        "Synthetic line mode set but no curve drawn (insufficient finite data)"
    )

if LOG_TIME_AXIS:
    lt = max(float(LOG_TIME_LINTHRESH_DAYS), 1e-6)
    ax.set_xscale("symlog", linthresh=lt, linscale=1.0, base=10)
    ax.set_xlabel(
        "Phase (days since %.5f MJD; symlog)" % t0,
        fontsize=16,
    )
else:
    ax.set_xlabel("Phase relative to merger (days)", fontsize=16)
ax.set_ylabel(ylabel, fontsize=16)
# ax.set_title(
#     r"%s: pseudo-$L_{\mathrm{bol}}$ ($\lambda=%.0f$--$%.0f$ \AA)"
#     % (SNNAME, lam_min_eff, lam_max_eff)
# )
ax.grid(True, which="major", linestyle="-", linewidth=0.8, alpha=0.7)
ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.5)
ax.tick_params(axis="both", which="major", labelsize=14)
h, lbl = ax.get_legend_handles_labels()
if h:
    ax.legend(loc="best", fontsize=16)
fig.tight_layout()
_maybe_save_fig(fig, FIG_SAVE_PATH)
plt.show()
